# Single-step rank activation

Companion notebook for Fig. 3a. Explore the cached structures or scores used in these panels; this notebook does not reproduce the final figure layout.


## What this demonstrates

A parent point group fixes the maximum rank available at each representation degree. A controlled deformation releases only the directions admitted by the child group, so the corresponding singular values rise from the numerical floor.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "demo" / "data" / "manifest.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the SymFP repository or its demo directory.")
CODE_ROOT = PROJECT_ROOT / "code" if (PROJECT_ROOT / "code" / "utils" / "demo_utils.py").is_file() else PROJECT_ROOT
sys.path.insert(0, str(CODE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from utils.demo_utils import (
    ANGULAR_MODES,
    DEGREE_STYLES,
    PALETTE,
    RANK_ACTIVATION_CASES,
    SUBGROUP_BOUNDARIES,
    SUBGROUP_COLUMNS,
    SUBGROUP_ENTERED,
    SUBGROUP_EXPECTED_ONSETS,
    SUBGROUP_TARGETS,
    environment_versions,
    load_angular_mode,
    load_rank_activation,
    load_subgroup_chain,
    runtime_summary,
)

runtime = runtime_summary()
display(environment_versions().style.hide(axis="index"))
print(f"Project root: {runtime['project_root']}")
print(f"Device available to the project: {runtime['device_available']}")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#555555",
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
})

## Configuration

Choose one published single-step deformation. Re-run the notebook after changing `CASE`.

In [ ]:
CASE = "B1_octahedron_tetragonal"
RANK_TOL = 1e-4
assert CASE in RANK_ACTIVATION_CASES
config = RANK_ACTIVATION_CASES[CASE]
print(config["label"])
print(f"Absolute rank tolerance: {RANK_TOL:g}")

## Load data

The notebook reads the bundled controlled-deformation structures and the cached parameter-free singular values used for Fig. 3. It does not write files or rerun a trained model.

In [ ]:
positions, frame_groups, spectra = load_rank_activation(CASE)
print(f"Loaded {len(positions)} structures and {len(spectra):,} singular values.")

## Plot

The left pair shows the parent and final structures. The right pair follows each singular value across the deformation. Teal is always allowed, coral is newly activated by symmetry breaking, and dashed purple remains forbidden.

In [ ]:
def infer_edges(coords: np.ndarray, scale: float = 1.18):
    """Infer only the nearest-neighbour skeleton for visualisation."""
    diff = coords[:, None, :] - coords[None, :, :]
    distances = np.linalg.norm(diff, axis=-1)
    distances[distances == 0] = np.inf
    cutoff = scale * np.median(distances.min(axis=1))
    return [(i, j) for i in range(len(coords)) for j in range(i + 1, len(coords))
            if distances[i, j] <= cutoff]


def set_equal_3d(ax, coords: np.ndarray):
    centre = coords.mean(axis=0)
    radius = np.ptp(coords, axis=0).max() / 2
    radius = max(radius, 1e-6)
    ax.set_xlim(centre[0] - radius, centre[0] + radius)
    ax.set_ylim(centre[1] - radius, centre[1] + radius)
    ax.set_zlim(centre[2] - radius, centre[2] + radius)
    ax.set_box_aspect((1, 1, 1))


def draw_structure(ax, coords: np.ndarray, title: str):
    for i, j in infer_edges(coords):
        ax.plot(*zip(coords[i], coords[j]), color=PALETTE["grey"], lw=1.2, zorder=1)
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], s=32, facecolor="white",
               edgecolor=PALETTE["ink"], linewidth=1.0, depthshade=False, zorder=2)
    set_equal_3d(ax, coords)
    ax.view_init(elev=24, azim=-58)
    ax.set_title(title, pad=6)
    ax.set_axis_off()

In [ ]:
fig = plt.figure(figsize=(11, 3.2), constrained_layout=True)
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1.35, 1.35])

parent_pos = positions[0]
child_pos = positions[-1]
draw_structure(fig.add_subplot(gs[0], projection="3d"), parent_pos, config["parent"])
draw_structure(fig.add_subplot(gs[1], projection="3d"), child_pos, config["child"])

category_style = {
    "always allowed": (PALETTE["teal"], "-", 1.7),
    "activated": (PALETTE["coral"], "-", 2.2),
    "forbidden": (PALETTE["purple"], "--", 1.1),
}

for panel, (degree, (parent_rank, child_rank)) in enumerate(config["degrees"].items(), start=2):
    ax = fig.add_subplot(gs[panel])
    subset = spectra[spectra["degree"] == degree]
    for index in sorted(subset["idx"].unique()):
        curve = subset[subset["idx"] == index].sort_values("frame")
        if index < parent_rank:
            category = "always allowed"
        elif index < child_rank:
            category = "activated"
        else:
            category = "forbidden"
        colour, linestyle, width = category_style[category]
        ax.plot(curve["epsilon"], np.maximum(curve["sigma"], 1e-12),
                color=colour, ls=linestyle, lw=width, alpha=0.95)
    ax.set_yscale("log")
    ax.set_title(f"$l={degree}$: rank {parent_rank} → {child_rank}")
    ax.set_xlabel("deformation parameter" if CASE != "B3_prism_axial_chirality" else "twist parameter")
    if panel == 2:
        ax.set_ylabel("singular value")

handles = [plt.Line2D([0], [0], color=v[0], ls=v[1], lw=v[2], label=k)
           for k, v in category_style.items()]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False,
           bbox_to_anchor=(0.72, 1.08))
plt.show()

In [ ]:
rows = []
for degree, (expected_parent, expected_child) in config["degrees"].items():
    subset = spectra[spectra["degree"] == degree]
    measured = {}
    for frame_name, frame in (("parent", 0), ("child", int(subset["frame"].max()))):
        values = subset[subset["frame"] == frame]["sigma"].to_numpy()
        measured[frame_name] = int((values > RANK_TOL).sum())
    rows.append({
        "degree": degree,
        "upper bound: parent": expected_parent,
        "measured: parent": measured["parent"],
        "upper bound: child": expected_child,
        "measured: child": measured["child"],
        "match": measured["parent"] == expected_parent and measured["child"] == expected_child,
    })
rank_table = pd.DataFrame(rows)
display(rank_table.style.hide(axis="index"))

## Short conclusion

In [ ]:
if rank_table["match"].all():
    print("PASS — the measured parent and child ranks match the fixed-subspace upper bounds at every displayed degree for this example.")
else:
    print("CHECK — at least one measured rank differs from the fixed-subspace upper bound at the selected tolerance; saturation is not guaranteed in general.")
